### Аналитическое решение линейной регрессии 

Следовательно:
$$
J(\theta) = \frac{1}{2} \left( y^\top y - 2\theta^\top X^\top y + \theta^\top X^\top X \theta \right)
$$

**Пояснение:**
- $X \in \mathbb{R}^{N \times d}$ — матрица признаков (матрица плана), где каждая строка соответствует вектору признаков $x_n^\top$,
- $y \in \mathbb{R}^N$ — вектор целевых значений,
- $\theta \in \mathbb{R}^d$ — вектор параметров (весов) модели.

Данное выражение представляет собой матрично-векторную запись функции потерь метода наименьших квадратов (МНК) с множителем $\frac{1}{2}$. Исходя из критерия суммы квадратов ошибок $L(\theta) = \sum_{n=1}^N (x_n^\top \theta - y_n)^2$, его можно компактно переписать в виде квадрата евклидовой нормы:
$$
L(\theta) = \|X\theta - y\|^2 = (X\theta - y)^\top (X\theta - y).
$$
Раскрывая произведение, получаем:
$$
\theta^\top X^\top X \theta - \theta^\top X^\top y - y^\top X \theta + y^\top y.
$$
Поскольку $\theta^\top X^\top y$ является скаляром, он равен своему транспонированному значению $y^\top X \theta$, поэтому два средних члена объединяются в $-2\theta^\top X^\top y$.

Множитель $\frac{1}{2}$ вводится исключительно для математического удобства: при вычислении градиента по $\theta$ он сокращает двойку, в результате чего получается:
$$
\nabla_\theta J(\theta) = X^\top X \theta - X^\top y.
$$
Приравнивая градиент к нулю, приходим к системе нормальных уравнений $X^\top X \theta = X^\top y$, решение которой в замкнутой форме (при условии обратимости $X^\top X$) даёт МНК-оценку:
$$
\theta^* = (X^\top X)^{-1} X^\top y.
$$

### Изменения при добавлении L1- и L2-регуляризации

Регуляризация добавляет штраф за сложность модели к функции потерь: $J_{\text{reg}}(w) = J(w) + \lambda R(w)$, где $\lambda > 0$ — коэффициент регуляризации.

| Тип | Функция потерь | Аналитическое решение | Поведение весов |
|-----|----------------|------------------------|-----------------|
| **L2 (Ridge)** | $\frac{1}{2}\|Xw-y\|^2 + \frac{\lambda}{2}\|w\|^2_2$ | $\displaystyle w^* = (X^\top X + \lambda I)^{-1} X^\top y$ | Гладкое сжатие к нулю, все веса $\neq 0$ |
| **L1 (Lasso)** | $\frac{1}{2}\|Xw-y\|^2 + \lambda \|w\|_1$ | **Замкнутой формы нет** | Точный ноль у части весов (разреженность) |

**Почему нет аналитического решения для L1?**  
Норма $\|w\|_1 = \sum |w_j|$ не дифференцируема в точке $w_j = 0$. Оптимальные условия записываются через субградиент:
$$
0 \in X^\top(Xw - y) + \lambda \, \partial \|w\|_1
$$
Решение находят итеративно (координатный спуск, проксимальный градиент, ADMM).



### Почему L1-регуляризация используется для отбора признаков?

1. **Геометрическая причина:**  
   Множество допустимых значений $\{w : \|w\|_1 \leq C\}$ имеет форму ромба/октаэдра с острыми вершинами на осях координат. Контур функции потерь (эллипсоид) чаще всего касается такого множества именно в вершине или на грани, где одна или несколько компонент $w_j = 0$.

2. **Алгоритмическая причина (мягкий порог):**  
   При решении Lasso через проксимальный оператор возникает операция *soft-thresholding*:
   $$
   w_j^{\text{new}} = \text{sign}(z_j) \cdot \max\left(0, |z_j| - \lambda\right)
   $$
   Если «сырое» значение $|z_j|$ меньше порога $\lambda$, вес обнуляется **точно**, а не асимптотически.

3. **Практический эффект:**  
   - Автоматический отбор наиболее информативных признаков,
   - Упрощение интерпретации модели,
   - Устойчивость к мультиколлинеарности (выбирает один признак из группы коррелированных).


### 4. Учёт нелинейных зависимостей в линейных моделях

Линейная модель остаётся **линейной по параметрам**, но может аппроксимировать нелинейности за счёт **нелинейного преобразования признаков**:
$$
f(x) = w^\top \phi(x)
$$
где $\phi: \mathbb{R}^d \to \mathbb{R}^M$ — функция отображения в пространство более высокой размерности.

**Основные подходы:**
- Полиномиальные признаки: $\phi(x) = [1, x_1, x_2, x_1^2, x_1x_2, x_2^2, \dots]$
- sin, exp, log
- Ядерный трюк (Kernel Trick):
  Вместо явного вычисления $\phi(x)$ вводится ядро $K(x_i, x_j) = \phi(x_i)^\top \phi(x_j)$. Решение ищется в виде $w = \sum_{n=1}^N \alpha_n \phi(x_n)$, а оптимизация сводится к задаче по $\alpha$ с матрицей Грама $K_{ij} = K(x_i, x_j)$.



In [2]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
from sklearn.preprocessing import PolynomialFeatures
warnings.filterwarnings('ignore')

In [3]:
df0 = pd.read_json('train.json')

In [4]:
features = (df0['features']
            .astype(str)
            .str.replace('[', '')
            .str.replace(']', '')
            .str.replace("'", '')
            .str.replace('"', '')
            .str.strip()
            .str.split(', ')
            .explode())
features = features[features != '']
len(features.unique().tolist())

1554

In [5]:
counter = Counter(features)
top = counter.most_common(20)
top

[('Elevator', 25915),
 ('Cats Allowed', 23540),
 ('Hardwood Floors', 23527),
 ('Dogs Allowed', 22035),
 ('Doorman', 20898),
 ('Dishwasher', 20426),
 ('No Fee', 18062),
 ('Laundry in Building', 16344),
 ('Fitness Center', 13252),
 ('Pre-War', 9148),
 ('Laundry in Unit', 8738),
 ('Roof Deck', 6542),
 ('Outdoor Space', 5268),
 ('Dining Room', 5136),
 ('High Speed Internet', 4299),
 ('Balcony', 2992),
 ('Swimming Pool', 2730),
 ('Laundry In Building', 2593),
 ('New Construction', 2559),
 ('Terrace', 2283)]

In [6]:
top_features = [name for name, count in top]
df = df0[['bathrooms', 'bedrooms', 'features']]
for feat in top_features:
    df[feat] = df['features'].apply(lambda x: 1 if feat in x else 0)

feature_list = top_features + ['bathrooms', 'bedrooms']
print(f"{len(feature_list)} признака(ов)")
print(feature_list)
df = df.drop('features', axis=1)
df.head(3)

22 признака(ов)
['Elevator', 'Cats Allowed', 'Hardwood Floors', 'Dogs Allowed', 'Doorman', 'Dishwasher', 'No Fee', 'Laundry in Building', 'Fitness Center', 'Pre-War', 'Laundry in Unit', 'Roof Deck', 'Outdoor Space', 'Dining Room', 'High Speed Internet', 'Balcony', 'Swimming Pool', 'Laundry In Building', 'New Construction', 'Terrace', 'bathrooms', 'bedrooms']


,bathrooms,bedrooms,Elevator,Cats Allowed,Hardwood Floors,Dogs Allowed,Doorman,Dishwasher,No Fee,Laundry in Building,...,Laundry in Unit,Roof Deck,Outdoor Space,Dining Room,High Speed Internet,Balcony,Swimming Pool,Laundry In Building,New Construction,Terrace
4,1.0,1,0,1,1,1,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0
6,1.0,2,1,0,1,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,1,0,1,0,1,1,0,1,...,1,0,0,0,0,0,0,0,0,0


In [7]:
class MyLinearRegression:
    def __init__(self, lr=0.01, epochs=100, seed=21, alpha_l1=0.0,
                 alpha_l2=0.0, reg_type='none', batch_size=1):
        self.lr = lr
        self.epochs = epochs
        self.seed = seed                
        self.alpha_l1 = alpha_l1
        self.alpha_l2 = alpha_l2
        self.reg_type = reg_type        
        self.batch_size = batch_size
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).flatten()
        n_samples, n_features = X.shape
        
        rng = np.random.default_rng(self.seed)
        self.coef_ = rng.normal(0, 0.01, n_features)
        self.intercept_ = 0.0
        
        for _ in range(self.epochs):
            indeces = rng.permutation(n_samples)
            
            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size
                idx = indeces[start:end]
                X_batch, y_batch = X[idx], y[idx]
                
                y_pred = X_batch @ self.coef_ + self.intercept_
                error = y_pred - y_batch
                
                grad_w = 2 * (X_batch.T @ error) / len(idx)
                grad_b = 2 * np.mean(error) 
                
                reg_grad_w = np.zeros_like(self.coef_)
                if self.reg_type == 'l1':
                    reg_grad_w = self.alpha_l1 * np.sign(self.coef_)
                elif self.reg_type == 'l2':
                    reg_grad_w = self.alpha_l2 * self.coef_
                elif self.reg_type == 'elasticnet':
                    reg_grad_w = self.alpha_l1 * np.sign(self.coef_) + self.alpha_l2 * self.coef_
                    
                self.coef_ -= self.lr * (grad_w + reg_grad_w)
                self.intercept_ -= self.lr * grad_b
            
        return self
            
    def predict(self, X):      
        X = np.asarray(X, dtype=np.float64)
        return X @ self.coef_ + self.intercept_
    
    @staticmethod
    def r2_score_custom(y_true, y_pred):
        res = np.sum((y_true - y_pred) ** 2)
        tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - (res / tot)

In [8]:
X = df.values
y = df0['price'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

In [21]:
def run_comparison_with_reg(X_train, y_train, X_test, y_test, reg_alpha=0.1):
    
    models = {
        'My Linear Regression': MyLinearRegression(lr=0.01, epochs=1000, seed=21, batch_size=1, reg_type='none'),
        'My Ridge': MyLinearRegression(lr=0.01, epochs=1000, seed=21, batch_size=1, reg_type='l2', alpha_l2=reg_alpha),
        'My Lasso': MyLinearRegression(lr=0.01, epochs=1000, seed=21, batch_size=1, reg_type='l1', alpha_l1=reg_alpha),
        'My ElasticNet': MyLinearRegression(lr=0.01, epochs=1000, seed=21, batch_size=1, reg_type='elasticnet', alpha_l1=reg_alpha),
        'Sklearn Linear Regression': LinearRegression(),
        'Sklearn Ridge': Ridge(alpha=reg_alpha),
        'Sklearn Lasso': Lasso(alpha=reg_alpha),
        'Sklern ElasticNet': ElasticNet(alpha=reg_alpha)
    }

    results = {'MAE': [], 'RMSE': [], 'R2': []}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_tr_pred = model.predict(X_train)
        y_te_pred = model.predict(X_test)

        results['MAE'].append({
            'model': name,
            'train': mean_absolute_error(y_train, y_tr_pred),
            'test': mean_absolute_error(y_test, y_te_pred)
        })
        results['RMSE'].append({
            'model': name,
            'train': root_mean_squared_error(y_train, y_tr_pred),
            'test': root_mean_squared_error(y_test, y_te_pred)
        })
        results['R2'].append({
            'model': name,
            'train': r2_score(y_train, y_tr_pred),
            'test': r2_score(y_test, y_te_pred)
        })

    mae_df = pd.DataFrame(results['MAE'])
    rmse_df = pd.DataFrame(results['RMSE'])
    r2_df = pd.DataFrame(results['R2'])

    return mae_df, rmse_df, r2_df

In [22]:
mae_table, rmse_table, r2_table = run_comparison_with_reg(X_train, y_train, X_test, y_test, reg_alpha=0.1)

print("MAE:\n", mae_table)
print("\nRMSE:\n", rmse_table)
print("\nR²:\n", r2_table)

MAE:
                        model        train         test
0       My Linear Regression  1155.710002   964.079555
1                   My Ridge  1085.515248   887.397543
2                   My Lasso  1155.607278   963.959919
3              My ElasticNet  1155.607278   963.959919
4  Sklearn Linear Regression  1247.439141  1049.511765
5              Sklearn Ridge  1247.431700  1049.503539
6              Sklearn Lasso  1247.034581  1049.060622
7          Sklern ElasticNet  1173.154091   966.890332

RMSE:
                        model         train         test
0       My Linear Regression  24591.183625  2233.120248
1                   My Ridge  24590.823933  2234.339435
2                   My Lasso  24591.185278  2233.104461
3              My ElasticNet  24591.185278  2233.104461
4  Sklearn Linear Regression  24567.312851  2193.942261
5              Sklearn Ridge  24567.312851  2193.938115
6              Sklearn Lasso  24567.312936  2193.720402
7          Sklern ElasticNet  24569.338878 

In [23]:
r2_table

,model,train,test
0,My Linear Regression,0.003562,0.301562
1,My Ridge,0.003591,0.300799
2,My Lasso,0.003562,0.301572
3,My ElasticNet,0.003562,0.301572
4,Sklearn Linear Regression,0.005495,0.325854
5,Sklearn Ridge,0.005495,0.325856
6,Sklearn Lasso,0.005495,0.325990
7,Sklern ElasticNet,0.005331,0.343175


Нормализация признаков обязательна для методов, чувствительных к масштабу данных: градиентного спуска (чтобы обеспечить быструю сходимость), метрик расстояния (KNN, K-means, чтобы признаки с большим разбросом не доминировали) и регуляризации (чтобы штраф применялся к весам равномерно). Для древовидных моделей (Decision Tree, Random Forest) нормализация не требуется, так как они используют пороговые правила, инвариантные к монотонным преобразованиям.

**Формула MinMaxScaler** (приведение к диапазону $[0, 1]$):

$$
x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

Параметры $x_{\min}$ и $x_{\max}$ вычисляются только на обучающей выборке и затем применяются к тестовой, чтобы избежать утечки данных.

In [24]:
def min_max_normalize(X, min_val=0.0, max_val=1.0):
    X = np.array(X, dtype=float)
    mn, mx = X.min(axis=0), X.max(axis=0)
    denom = mx - mn
    denom[denom == 0] = 1.0
    return min_val + (X - mn) * (max_val - min_val) / denom

In [28]:
X_minmax_my = min_max_normalize(X)
X_minmax_sklearn = MinMaxScaler().fit_transform(X)
print(X_minmax_my)
print(f'\n\n\n{X_minmax_sklearn}')

[[0.1   0.125 0.    ... 0.    0.    0.   ]
 [0.1   0.25  1.    ... 0.    0.    0.   ]
 [0.1   0.25  1.    ... 0.    0.    0.   ]
 ...
 [0.1   0.125 1.    ... 0.    0.    0.   ]
 [0.1   0.25  0.    ... 0.    0.    0.   ]
 [0.1   0.375 1.    ... 0.    0.    0.   ]]



[[0.1   0.125 0.    ... 0.    0.    0.   ]
 [0.1   0.25  1.    ... 0.    0.    0.   ]
 [0.1   0.25  1.    ... 0.    0.    0.   ]
 ...
 [0.1   0.125 1.    ... 0.    0.    0.   ]
 [0.1   0.25  0.    ... 0.    0.    0.   ]
 [0.1   0.375 1.    ... 0.    0.    0.   ]]


### StandardScaler

**StandardScaler** приводит признаки к распределению с нулевым средним и единичной дисперсией:

$$
x_{\text{scaled}} = \frac{x - \mu}{\sigma}
$$

где $\mu$ — среднее, $\sigma$ — стандартное отклонение признака.

In [29]:
def my_standart_scaler(X):
    m, s = X.mean(axis=0), X.std(axis=0)
    s[s == 0] = 1.0
    return (X - m) / s

In [31]:
X_scaler_my = my_standart_scaler(X)
X_scaler_sklearn = StandardScaler().fit_transform(X)
print(X_scaler_my)
print(f'\n\n\n{X_scaler_sklearn}')

[[-0.42316255 -0.48577234 -1.05153709 ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  0.41108287  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  0.41108287  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]
 ...
 [-0.42316255 -0.48577234  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  0.41108287 -1.05153709 ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  1.30793808  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]]



[[-0.42316255 -0.48577234 -1.05153709 ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  0.41108287  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  0.41108287  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]
 ...
 [-0.42316255 -0.48577234  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  0.41108287 -1.05153709 ... -0.23548793 -0.23385394
  -0.22023456]
 [-0.42316255  1.30793808  0.9509888  ... -0.23548793 -0.23385394
  -0.22023456]]


In [34]:
def collect_metrics(model, name, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_tr, y_te = model.predict(X_train), model.predict(X_test)
    return {
        'model': name,
        'MAE_train': mean_absolute_error(y_train, y_tr), 'MAE_test': mean_absolute_error(y_test, y_te),
        'RMSE_train': root_mean_squared_error(y_train, y_tr), 'RMSE_test': root_mean_squared_error(y_test, y_te),
        'R2_train': r2_score(y_train, y_tr), 'R2_test': r2_score(y_test, y_te)
    }

all_results = []


In [40]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=21)
for m, n in [
    (MyLinearRegression(lr=0.01, epochs=500), 'Linreg default'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l2=0.1, reg_type='l2'), 'Ridge default'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l1=0.1, reg_type='l1'), 'Lasso default'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l1=0.1, alpha_l2=0.1, reg_type='elasticnet'), 'ElasticNet default')
]:
    all_results.append(collect_metrics(m, n, X_tr, X_te, y_tr, y_te))

In [41]:
X_tr, X_te, y_tr, y_te = train_test_split(X_minmax_my, y, test_size=0.2, random_state=21)
for m, n in [
    (MyLinearRegression(lr=0.01, epochs=500), 'Linear MinMaxScaler'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l2=0.1, reg_type='l2'), 'Ridge MinMaxScaler'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l1=0.1, reg_type='l1'), 'Lasso MinMaxScaler'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l1=0.1, alpha_l2=0.1, reg_type='elasticnet'), 'ElasticNet MinMaxScaler')
]:
    all_results.append(collect_metrics(m, n, X_tr, X_te, y_tr, y_te))

In [43]:
X_tr, X_te, y_tr, y_te = train_test_split(X_scaler_my, y, test_size=0.2, random_state=21)
for m, n in [
    (MyLinearRegression(lr=0.01, epochs=500), 'Linear StandardScaler'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l2=0.1, reg_type='l2'), 'Ridge StandardScaler'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l1=0.1, reg_type='l1'), 'Lasso StandardScaler'),
    (MyLinearRegression(lr=0.01, epochs=500, alpha_l1=0.1, alpha_l2=0.1, reg_type='elasticnet'), 'ElasticNet StandardScaler')
]:
    all_results.append(collect_metrics(m, n, X_tr, X_te, y_tr, y_te))

In [ ]:
final_df = pd.DataFrame(all_results)
print(final_df.round(4))

                        model  MAE_train   MAE_test  RMSE_train  RMSE_test  \
0              Linreg default  1155.7100   964.0796  24591.1836  2233.1202   
1               Ridge default  1085.5152   887.3975  24590.8239  2234.3394   
2               Lasso default  1155.6073   963.9599  24591.1853  2233.1045   
3              Linreg default  1070.7175   872.7594  24580.0354  2159.8494   
4               Ridge default  1043.0578   838.3872  24584.5187  2190.8598   
5               Lasso default  1070.5841   872.6032  24580.0439  2159.8520   
6          ElasticNet default  1043.0453   838.3710  24584.5257  2190.8996   
7         Linear MinMaxScaler  1101.3227   897.9222  24576.2483  2134.0210   
8          Ridge MinMaxScaler  1302.9972  1088.3497  24609.2266  2435.8534   
9          Lasso MinMaxScaler  1101.1232   897.6961  24576.2554  2134.0140   
10    ElasticNet MinMaxScaler  1303.0442  1088.3941  24609.2362  2435.9214   
11      Linear StandardScaler  1210.9373  1008.6876  24591.8544 

In [47]:
X_base = df[['bathrooms', 'bedrooms']].values
y = df0['price'].values

poly = PolynomialFeatures(degree=10, include_bias=False)
X_poly = poly.fit_transform(X_base)

In [48]:
scaler = StandardScaler()
X_poly_sc = scaler.fit_transform(X_poly)

X_train, X_test, y_train, y_test = train_test_split(X_poly_sc, y, test_size=0.2, random_state=42)

In [52]:
for m, n in [
    (LinearRegression(), 'Linear Poly10'),
    (Ridge(alpha=0.1), 'Ridge Poly10'),
    (Lasso(alpha=0.1), 'Lasso Poly10'),
    (ElasticNet(alpha=0.1), 'ElasticNet Poly10')
]:
    all_results.append(collect_metrics(m, n, X_train, X_test, y_train, y_test))

In [53]:
final_df = pd.DataFrame(all_results)
print(final_df.round(4))

                        model  MAE_train   MAE_test  RMSE_train   RMSE_test  \
0              Linreg default  1155.7100   964.0796  24591.1836   2233.1202   
1               Ridge default  1085.5152   887.3975  24590.8239   2234.3394   
2               Lasso default  1155.6073   963.9599  24591.1853   2233.1045   
3              Linreg default  1070.7175   872.7594  24580.0354   2159.8494   
4               Ridge default  1043.0578   838.3872  24584.5187   2190.8598   
5               Lasso default  1070.5841   872.6032  24580.0439   2159.8520   
6          ElasticNet default  1043.0453   838.3710  24584.5257   2190.8996   
7         Linear MinMaxScaler  1101.3227   897.9222  24576.2483   2134.0210   
8          Ridge MinMaxScaler  1302.9972  1088.3497  24609.2266   2435.8534   
9          Lasso MinMaxScaler  1101.1232   897.6961  24576.2554   2134.0140   
10    ElasticNet MinMaxScaler  1303.0442  1088.3941  24609.2362   2435.9214   
11      Linear StandardScaler  1210.9373  1008.6876 

In [54]:
summary = final_df.groupby('model')[['MAE_train', 'MAE_test', 'RMSE_train', 'RMSE_test', 'R2_train', 'R2_test']].agg(['mean', 'median']).round(4)
summary_flat = pd.DataFrame({
    f"{metric}_{stat}": summary[metric][stat] 
    for metric in summary.columns.get_level_values(0).unique() 
    for stat in ['mean', 'median']
}).reset_index()

final_df_with_summary = pd.concat([final_df, summary_flat], ignore_index=True)
print(summary_flat.to_string(index=False))

                    model  MAE_train_mean  MAE_train_median  MAE_test_mean  MAE_test_median  RMSE_train_mean  RMSE_train_median  RMSE_test_mean  RMSE_test_median  R2_train_mean  R2_train_median  R2_test_mean  R2_test_median
  ElasticNet MinMaxScaler       1303.0442         1303.0442      1088.3941        1088.3941       24609.2362         24609.2362       2435.9214         2435.9214         0.0021           0.0021        0.1689          0.1689
        ElasticNet Poly10        973.4043          973.4043      1344.5103        1344.5103        9706.1914          9706.1914      45196.9654        45196.9654         0.0313           0.0313        0.0013          0.0013
ElasticNet StandardScaler       1194.4551         1194.4551       992.1949         992.1949       24592.3867         24592.3867       2284.1393         2284.1393         0.0035           0.0035        0.2693          0.2693
       ElasticNet default       1043.0453         1043.0453       838.3710         838.3710       24584.

In [57]:
best_model = summary_flat.loc[summary_flat['R2_test_median'].idxmax()]
print(f"\nЛучшая модель по R²_test:")
print(f"   {best_model['model']}")
print(f"   R²_test: {best_model['R2_test_median']:.4f} | MAE_test: {best_model['MAE_test_median']:.2f}")

summary_flat['R2_gap'] = (summary_flat['R2_train_median'] - summary_flat['R2_test_median']).abs()
stable_model = summary_flat.loc[summary_flat['R2_gap'].idxmin()]
print(f"\n Стабильная модель (мин. разрыв R² train/test):")
print(f"   {stable_model['model']}")
print(f"   Разрыв R²: {stable_model['R2_gap']:.4f}")
print(f"   R²_train: {stable_model['R2_train_median']:.4f} | R²_test: {stable_model['R2_test_median']:.4f}")


Лучшая модель по R²_test:
   Lasso MinMaxScaler
   R²_test: 0.3622 | MAE_test: 897.70

 Стабильная модель (мин. разрыв R² train/test):
   ElasticNet Poly10
   Разрыв R²: 0.0300
   R²_train: 0.0313 | R²_test: 0.0013


In [73]:
y_clean = y[y > 0]
X_clean = X_scaler_my[y > 0]
y_log = np.log(y_clean)

X_train, X_test, y_train_log, y_test_log = train_test_split(X_clean, y_log, test_size=0.2, random_state=21)

In [74]:
model_log = Ridge()
model_log.fit(X_train, y_train_log)

y_pred_train = np.exp(model_log.predict(X_train))
y_pred_test = np.exp(model_log.predict(X_test))
y_test_orig = np.exp(y_test_log)
print(f"Log transform: MAE_test = {mean_absolute_error(y_test_orig, y_pred_test):.2f}, R²_test = {r2_score(y_test_orig, y_pred_test):.4f}")

Log transform: MAE_test = 785.72, R²_test = 0.4560


In [71]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X_scaler_my, y, test_size=0.2, random_state=21)

Q1, Q3 = np.percentile(y_train_raw, 25), np.percentile(y_train_raw, 75)
IQR = Q3 - Q1
lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR

mask = (y_train_raw >= lower) & (y_train_raw <= upper)
X_train_no_out, y_train_no_out = X_train_raw[mask], y_train_raw[mask]

In [72]:
model_no_out = Ridge()
model_no_out.fit(X_train_no_out, y_train_no_out)

y_pred_test_no_out = model_no_out.predict(X_test_raw)
print(f"No outliers: MAE_test = {mean_absolute_error(y_test_raw, y_pred_test_no_out):.2f}, R²_test = {r2_score(y_test_raw, y_pred_test_no_out):.4f}")


No outliers: MAE_test = 817.66, R²_test = 0.3223


In [67]:
class MyLinearRegressionBatch:
    def __init__(self, lr=0.01, epochs=500, seed=21, reg_type='none', alpha_l1=0.0, alpha_l2=0.0):
        self.lr, self.epochs, self.seed = lr, epochs, seed
        self.reg_type, self.alpha_l1, self.alpha_l2 = reg_type, alpha_l1, alpha_l2
        self.coef_, self.intercept_ = None, None

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float).flatten()
        rng = np.random.default_rng(self.seed)
        self.coef_ = rng.normal(0, 0.01, X.shape[1])
        self.intercept_ = 0.0
        for _ in range(self.epochs):
            y_pred = X @ self.coef_ + self.intercept_
            error = y_pred - y
            grad_w = 2 * (X.T @ error) / len(y)
            grad_b = 2 * np.mean(error)
            reg_grad = self.alpha_l2 * self.coef_ if self.reg_type == 'l2' else self.alpha_l1 * np.sign(self.coef_) if self.reg_type == 'l1' else 0
            self.coef_ -= self.lr * (grad_w + reg_grad)
            self.intercept_ -= self.lr * grad_b
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

In [68]:
model_batch = MyLinearRegressionBatch(lr=0.01, epochs=500, seed=21, reg_type='l2', alpha_l2=0.1)
model_batch.fit(X_train, y_train)
y_pred_batch_train = model_batch.predict(X_train)
y_pred_batch_test = model_batch.predict(X_test)
print(f"Batch GD: R²_train = {r2_score(y_train, y_pred_batch_train):.4f}, R²_test = {r2_score(y_test, y_pred_batch_test):.4f}")

Batch GD: R²_train = -0.0008, R²_test = 0.0001
